# Damage Detection
### Dataset: https://www.kaggle.com/datasets/lplenka/coco-car-damage-detection-dataset

#### Inspect data format

In [ ]:
import json

with open("../data/damagedetectiondataset1/train/COCO_mul_train_annos.json") as f:
    mul_annos = json.load(f)

print(mul_annos.keys())
print("\nCategories:")
print(mul_annos['categories'])
print("\nFirst image entry:")
print(mul_annos['images'][0])
print("\nFirst annotation entry:")
print(mul_annos['annotations'][0])

#### Explore Data

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

with open("../data/damagedetectiondataset1/train/COCO_mul_train_annos.json") as f:
    coco_data = json.load(f)

# First image entry and its corresponding annotations
img_info = coco_data['images'][0]
img_id = img_info['id']
annos = [anno for anno in coco_data['annotations'] if anno['image_id'] == img_id]
categories = {cat['id']: cat['name'] for cat in coco_data['categories']}

img = Image.open(f"../data/damagedetectiondataset1/train/{img_info['file_name']}")

fig, ax = plt.subplots(1, figsize=(10, 10))
ax.imshow(img)

for anno in annos:
    x, y, w, h = anno['bbox']
    rect = patches.Rectangle((x, y), w, h, linewidth=2, edgecolor='r', facecolor='none')
    ax.add_patch(rect)
    ax.text(x, y - 10, categories[anno['category_id']], color='red', fontsize=12, weight='bold')

    for seg in anno.get('segmentation', []):
        poly_points = [(seg[i], seg[i + 1]) for i in range(0, len(seg), 2)]
        polygon = patches.Polygon(poly_points, closed=True, linewidth=2, edgecolor='yellow', facecolor='none', alpha = 0.5)
        ax.add_patch(polygon)

plt.axis('off')
plt.show()

In [ ]:
with open("../data/damagedetectiondataset1/train/COCO_mul_train_annos.json") as f:
    coco_data = json.load(f)

categories = {c['id']: c['name'] for c in coco_data['categories']}
images_by_id = {img['id']: img for img in coco_data['images']}

# Group annotations by category so we can grab one example per part
annos_by_category = {}
for anno in coco_data['annotations']:
    cat_name = categories[anno['category_id']]
    annos_by_category.setdefault(cat_name, []).append(anno)

# Show one example from each of the 5 categories
fig, axes = plt.subplots(1, 5, figsize=(25, 6))

for ax, (cat_name, annos) in zip(axes, annos_by_category.items()):
    anno = annos[0]  # just grab the first example for this category
    img_info = images_by_id[anno['image_id']]
    img = Image.open(f"../data/damagedetectiondataset1/train/{img_info['file_name']}")

    ax.imshow(img)
    x, y, w, h = anno['bbox']
    rect = patches.Rectangle((x, y), w, h, linewidth=2, edgecolor='red', facecolor='none')
    ax.add_patch(rect)

    for seg in anno['segmentation']:
        poly_points = [(seg[i], seg[i+1]) for i in range(0, len(seg), 2)]
        polygon = patches.Polygon(poly_points, closed=True, edgecolor='yellow', facecolor='yellow', alpha=0.3)
        ax.add_patch(polygon)

    ax.set_title(cat_name)
    ax.axis('off')

plt.tight_layout()
plt.show()

#### Class balance check

In [ ]:
from collections import Counter

with open("../data/damagedetectiondataset1/train/COCO_mul_train_annos.json") as f:
    train_data = json.load(f)

with open("../data/damagedetectiondataset1/val/COCO_mul_val_annos.json") as f:
    val_data = json.load(f)

categories = {c['id']: c['name'] for c in train_data['categories']}

train_counts = Counter(categories[a['category_id']] for a in train_data['annotations'])
val_counts = Counter(categories[a['category_id']] for a in val_data['annotations'])

print("Train annotation counts:")
for cat, count in train_counts.items():
    print(f"  {cat}: {count}")

print(f"\nTotal train annotations: {sum(train_counts.values())}")
print(f"Total train images: {len(train_data['images'])}")

print("\nVal annotation counts:")
for cat, count in val_counts.items():
    print(f"  {cat}: {count}")

print(f"\nTotal val annotations: {sum(val_counts.values())}")
print(f"Total val images: {len(val_data['images'])}")

#### Notes
-Average annotations per image is high (177 annotations / 59 images), most images have multiple damaged parts labeled  
-Headlamp is noticeably overrepresented  
-Val set is thin  
-Small dataset and clear imbalance, worth noting for now but will continue with training

In [ ]:
import os
import numpy as np
from PIL import ImageDraw
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as F

def polygon_to_mask(segmentation, height, width):
    """Converts a COCO polygon segmentation into a binary mask."""
    mask = Image.new('L', (width, height), 0)
    for polygon in segmentation:
        points = [(polygon[i], polygon[i + 1]) for i in range(0, len(polygon), 2)]
        ImageDraw.Draw(mask).polygon(points, outline=1, fill=1)
    return np.array(mask, dtype=np.uint8)


class DamageDataset(Dataset):
    def __init__(self, coco_data, img_dir, transform=None):
        self.img_dir = img_dir
        self.transform = transform

        self.images = {img['id']: img for img in coco_data['images']}
        self.categories = {cat['id']: cat['name'] for cat in coco_data['categories']}

        self.annotations_by_image = {}
        for anno in coco_data['annotations']:
            self.annotations_by_image.setdefault(anno['image_id'], []).append(anno)

        self.image_ids = [img_id for img_id in self.images if img_id in self.annotations_by_image]

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        image_id = self.image_ids[idx]
        img_info = self.images[image_id]
        annos = self.annotations_by_image[image_id]

        img_path = os.path.join(self.img_dir, img_info['file_name'])
        img = Image.open(img_path).convert("RGB")
        width, height = img.size

        boxes, labels, masks, areas, iscrowd = [], [], [], [], []

        for anno in annos:
            x, y, w, h = anno['bbox']
            boxes.append([x, y, x + w, y + h])
            labels.append(anno['category_id'])
            masks.append(polygon_to_mask(anno['segmentation'], height, width))
            areas.append(anno['area'])
            iscrowd.append(anno.get('iscrowd', 0))

        target = {
            'boxes': torch.as_tensor(boxes, dtype=torch.float32),
            'labels': torch.as_tensor(labels, dtype=torch.int64),
            'masks': torch.as_tensor(np.array(masks), dtype=torch.uint8),
            'image_id': torch.tensor([image_id]),
            'area': torch.as_tensor(areas, dtype=torch.float32),
            'iscrowd': torch.as_tensor(iscrowd, dtype=torch.int64),
        }

        if self.transform:
            img = self.transform(img)
        else:
            img = F.to_tensor(img)

        return img, target

In [ ]:
with open("../data/damagedetectiondataset1/train/COCO_mul_train_annos.json") as f:
    train_coco = json.load(f)

train_dataset = DamageDataset(train_coco, "../data/damagedetectiondataset1/train")

print(f"Number of training images (with annotations): {len(train_dataset)}")

img, target = train_dataset[0]
print(f"\nImage shape: {img.shape}")
print(f"Target keys: {target.keys()}")
print(f"Boxes: {target['boxes']}")
print(f"Labels: {target['labels']}")
print(f"Masks shape: {target['masks'].shape}")

In [ ]:
print(train_dataset.categories[target['labels'][0].item()])

#### Data Loader

In [ ]:
def collate_fn(batch):
    return tuple(zip(*batch))


with open("../data/damagedetectiondataset1/val/COCO_mul_val_annos.json") as f:
    val_coco = json.load(f)

val_dataset = DamageDataset(val_coco, "../data/damagedetectiondataset1/val")

train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=2, shuffle=False, collate_fn=collate_fn)

images, targets = next(iter(train_loader))
print(f"Batch size: {len(images)}")
print(f"First image shape: {images[0].shape}")
print(f"First target boxes: {targets[0]['boxes']}")
print(f"Second target boxes: {targets[1]['boxes']}")

In [ ]:
import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor

def build_model(num_classes):
    # num_classes should include background, so 5 parts + 1 background = 6
    model = torchvision.models.detection.maskrcnn_resnet50_fpn(weights="DEFAULT")

    # Swap the box classification head
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

    # Swap the mask prediction head
    in_features_mask = model.roi_heads.mask_predictor.conv5_mask.in_channels
    hidden_layer = 256
    model.roi_heads.mask_predictor = MaskRCNNPredictor(in_features_mask, hidden_layer, num_classes)

    return model


num_classes = len(train_dataset.categories) + 1  # +1 for background
model = build_model(num_classes)
print(f"Num classes (including background): {num_classes}")

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = model.to(device)

params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)

#### Training Loop

In [ ]:
def train_one_epoch(model, optimizer, data_loader, device):
    model.train()
    total_loss = 0.0

    for images, targets in data_loader:
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())

        optimizer.zero_grad()
        losses.backward()
        optimizer.step()

        total_loss += losses.item()

    return total_loss / len(data_loader)

"""
num_epochs = 10

for epoch in range(num_epochs):
    train_loss = train_one_epoch(model, optimizer, train_loader, device)
    print(f"Epoch {epoch+1}/{num_epochs} | Train Loss: {train_loss:.4f}")
    """

In [ ]:
def evaluate_loss(model, data_loader, device):
    model.train() 
    total_loss = 0.0

    with torch.no_grad():
        for images, targets in data_loader:
            images = [img.to(device) for img in images]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

            loss_dict = model(images, targets)
            losses = sum(loss for loss in loss_dict.values())
            total_loss += losses.item()

    return total_loss / len(data_loader)


val_loss = evaluate_loss(model, val_loader, device)
print(f"Val Loss: {val_loss:.4f}")

#### Notes:
-Clear Val Loss difference (1.1626 vs 0.2484)  
-Signal of overfitting, since a very small dataset

Look at actual predictions.

In [ ]:
model.eval()

img, target = val_dataset[0]
with torch.no_grad():
    prediction = model([img.to(device)])

print(prediction[0].keys())
print(f"Predicted boxes: {prediction[0]['boxes']}")
print(f"Predicted labels: {prediction[0]['labels']}")
print(f"Predicted scores: {prediction[0]['scores']}")

In [ ]:
img, target = val_dataset[0]
img_display = img.permute(1, 2, 0).numpy()  # convert from [C,H,W] to [H,W,C] for plotting

confidence_threshold = 0.5
keep = prediction[0]['scores'] > confidence_threshold

pred_boxes = prediction[0]['boxes'][keep].cpu()
pred_labels = prediction[0]['labels'][keep].cpu()
pred_scores = prediction[0]['scores'][keep].cpu()

fig, axes = plt.subplots(1, 2, figsize=(20, 10))

axes[0].imshow(img_display)
axes[0].set_title("Actual")
for box, label in zip(target['boxes'], target['labels']):
    x1, y1, x2, y2 = box
    rect = patches.Rectangle((x1, y1), x2-x1, y2-y1, linewidth=2, edgecolor='lime', facecolor='none')
    axes[0].add_patch(rect)
    axes[0].text(x1, y1-10, train_dataset.categories[label.item()], color='lime', fontsize=12, weight='bold')
axes[0].axis('off')

# Predictions (only confidence > threshold)
axes[1].imshow(img_display)
axes[1].set_title(f"Predictions (score > {confidence_threshold})")
for box, label, score in zip(pred_boxes, pred_labels, pred_scores):
    x1, y1, x2, y2 = box
    rect = patches.Rectangle((x1, y1), x2-x1, y2-y1, linewidth=2, edgecolor='red', facecolor='none')
    axes[1].add_patch(rect)
    cat_name = train_dataset.categories[label.item()]
    axes[1].text(x1, y1-10, f"{cat_name} {score:.2f}", color='red', fontsize=12, weight='bold')
axes[1].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
confidence_threshold = 0.7  # raised from 0.5

model.eval()
num_to_check = 5

fig, axes = plt.subplots(num_to_check, 2, figsize=(16, 8 * num_to_check))

for i in range(num_to_check):
    img, target = val_dataset[i]
    with torch.no_grad():
        prediction = model([img.to(device)])

    img_display = img.permute(1, 2, 0).numpy()
    keep = prediction[0]['scores'] > confidence_threshold
    pred_boxes = prediction[0]['boxes'][keep].cpu()
    pred_labels = prediction[0]['labels'][keep].cpu()
    pred_scores = prediction[0]['scores'][keep].cpu()

    # Ground truth
    axes[i, 0].imshow(img_display)
    axes[i, 0].set_title(f"Image {i} - Ground Truth")
    for box, label in zip(target['boxes'], target['labels']):
        x1, y1, x2, y2 = box
        rect = patches.Rectangle((x1, y1), x2-x1, y2-y1, linewidth=2, edgecolor='lime', facecolor='none')
        axes[i, 0].add_patch(rect)
        axes[i, 0].text(x1, y1-10, train_dataset.categories[label.item()], color='lime', fontsize=10, weight='bold')
    axes[i, 0].axis('off')

    # Predictions
    axes[i, 1].imshow(img_display)
    axes[i, 1].set_title(f"Image {i} - Predictions (score > {confidence_threshold})")
    for box, label, score in zip(pred_boxes, pred_labels, pred_scores):
        x1, y1, x2, y2 = box
        rect = patches.Rectangle((x1, y1), x2-x1, y2-y1, linewidth=2, edgecolor='red', facecolor='none')
        axes[i, 1].add_patch(rect)
        cat_name = train_dataset.categories[label.item()]
        axes[i, 1].text(x1, y1-10, f"{cat_name} {score:.2f}", color='red', fontsize=10, weight='bold')
    axes[i, 1].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
import torchvision.transforms.v2 as T
from torchvision import tv_tensors


def get_transform(train):
    transforms = []
    transforms.append(T.ToImage())
    if train:
        transforms.append(T.RandomHorizontalFlip(0.5))
        transforms.append(T.RandomPhotometricDistort(p=0.5))  # brightness/contrast/saturation jitter
    transforms.append(T.ToDtype(torch.float32, scale=True))
    return T.Compose(transforms)

In [ ]:
def __getitem__(self, idx):
    image_id = self.image_ids[idx]
    img_info = self.images[image_id]
    annos = self.annotations_by_image[image_id]

    img_path = os.path.join(self.img_dir, img_info['file_name'])
    img = Image.open(img_path).convert("RGB")  # stays as PIL for now
    width, height = img.size

    boxes, labels, masks, areas, iscrowd = [], [], [], [], []

    for anno in annos:
        x, y, w, h = anno['bbox']
        boxes.append([x, y, x + w, y + h])
        labels.append(anno['category_id'])
        masks.append(polygon_to_mask(anno['segmentation'], height, width))
        areas.append(anno['area'])
        iscrowd.append(anno.get('iscrowd', 0))

    target = {
        'boxes': tv_tensors.BoundingBoxes(boxes, format="XYXY", canvas_size=(height, width)),
        'labels': torch.as_tensor(labels, dtype=torch.int64),
        'masks': tv_tensors.Mask(torch.as_tensor(np.array(masks), dtype=torch.uint8)),
        'image_id': torch.tensor([image_id]),
        'area': torch.as_tensor(areas, dtype=torch.float32),
        'iscrowd': torch.as_tensor(iscrowd, dtype=torch.int64),
    }

    if self.transform:
        img, target = self.transform(img, target)

    return img, target

In [ ]:
train_dataset = DamageDataset(train_coco, "../data/damagedetectiondataset1/train", transform=get_transform(train=True))
val_dataset = DamageDataset(val_coco, "../data/damagedetectiondataset1/val", transform=get_transform(train=False))

train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=2, shuffle=False, collate_fn=collate_fn)

In [ ]:
images, targets = next(iter(train_loader))
print(images[0].shape, images[0].dtype)
print(targets[0]['boxes'])

In [ ]:
# Grab the same image twice — augmentation is random, so two pulls should differ
img1, target1 = train_dataset[0]
img2, target2 = train_dataset[0]

fig, axes = plt.subplots(1, 2, figsize=(16, 8))
axes[0].imshow(img1.permute(1, 2, 0))
axes[0].set_title(f"Sample 1 - boxes: {target1['boxes'].shape[0]}")
axes[0].axis('off')

axes[1].imshow(img2.permute(1, 2, 0))
axes[1].set_title(f"Sample 2 - boxes: {target2['boxes'].shape[0]}")
axes[1].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 6, figsize=(24, 5))
for i, ax in enumerate(axes):
    img, target = train_dataset[0]
    ax.imshow(img.permute(1, 2, 0))
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
model = build_model(num_classes)
model = model.to(device)

params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)

num_epochs = 10

for epoch in range(num_epochs):
    train_loss = train_one_epoch(model, optimizer, train_loader, device)
    val_loss = evaluate_loss(model, val_loader, device)
    print(f"Epoch {epoch+1}/{num_epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

In [ ]:
model = build_model(num_classes)
model = model.to(device)

params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)

os.makedirs("../models/damage_detector", exist_ok=True)
best_val_loss = float('inf')
num_epochs = 10

for epoch in range(num_epochs):
    train_loss = train_one_epoch(model, optimizer, train_loader, device)
    val_loss = evaluate_loss(model, val_loader, device)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "../models/damage_detector/maskrcnn_damage_best.pth")
        print(f"  -> New best model saved (Val Loss: {val_loss:.4f})")

    print(f"Epoch {epoch+1}/{num_epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

In [ ]:
model = build_model(num_classes)
model.load_state_dict(torch.load("../models/damage_detector/maskrcnn_damage_best.pth"))
model = model.to(device)

val_loss = evaluate_loss(model, val_loader, device)
print(f"Best checkpoint - Val Loss: {val_loss:.4f}")

In [ ]:
model.eval()
num_to_check = 5

fig, axes = plt.subplots(num_to_check, 2, figsize=(16, 8 * num_to_check))
confidence_threshold = 0.7

for i in range(num_to_check):
    img, target = val_dataset[i]
    with torch.no_grad():
        prediction = model([img.to(device)])

    img_display = img.permute(1, 2, 0).cpu().numpy()
    keep = prediction[0]['scores'] > confidence_threshold
    pred_boxes = prediction[0]['boxes'][keep].cpu()
    pred_labels = prediction[0]['labels'][keep].cpu()
    pred_scores = prediction[0]['scores'][keep].cpu()

    axes[i, 0].imshow(img_display)
    axes[i, 0].set_title(f"Image {i} - Ground Truth")
    for box, label in zip(target['boxes'], target['labels']):
        x1, y1, x2, y2 = box
        rect = patches.Rectangle((x1, y1), x2-x1, y2-y1, linewidth=2, edgecolor='lime', facecolor='none')
        axes[i, 0].add_patch(rect)
        axes[i, 0].text(x1, y1-10, train_dataset.categories[label.item()], color='lime', fontsize=10, weight='bold')
    axes[i, 0].axis('off')

    axes[i, 1].imshow(img_display)
    axes[i, 1].set_title(f"Image {i} - Predictions (score > {confidence_threshold})")
    for box, label, score in zip(pred_boxes, pred_labels, pred_scores):
        x1, y1, x2, y2 = box
        rect = patches.Rectangle((x1, y1), x2-x1, y2-y1, linewidth=2, edgecolor='red', facecolor='none')
        axes[i, 1].add_patch(rect)
        cat_name = train_dataset.categories[label.item()]
        axes[i, 1].text(x1, y1-10, f"{cat_name} {score:.2f}", color='red', fontsize=10, weight='bold')
    axes[i, 1].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
img, target = val_dataset[2]
with torch.no_grad():
    prediction = model([img.to(device)])

print(prediction[0]['scores'])
print(prediction[0]['labels'])

In [ ]:
confidence_threshold = 0.5

img, target = val_dataset[2]
with torch.no_grad():
    prediction = model([img.to(device)])

img_display = img.permute(1, 2, 0).cpu().numpy()
keep = prediction[0]['scores'] > confidence_threshold
pred_boxes = prediction[0]['boxes'][keep].cpu()
pred_labels = prediction[0]['labels'][keep].cpu()
pred_scores = prediction[0]['scores'][keep].cpu()

fig, axes = plt.subplots(1, 2, figsize=(20, 10))

axes[0].imshow(img_display)
axes[0].set_title("Ground Truth")
for box, label in zip(target['boxes'], target['labels']):
    x1, y1, x2, y2 = box
    rect = patches.Rectangle((x1, y1), x2-x1, y2-y1, linewidth=2, edgecolor='lime', facecolor='none')
    axes[0].add_patch(rect)
    axes[0].text(x1, y1-10, train_dataset.categories[label.item()], color='lime', fontsize=10, weight='bold')
axes[0].axis('off')

axes[1].imshow(img_display)
axes[1].set_title(f"Predictions (score > {confidence_threshold})")
for box, label, score in zip(pred_boxes, pred_labels, pred_scores):
    x1, y1, x2, y2 = box
    rect = patches.Rectangle((x1, y1), x2-x1, y2-y1, linewidth=2, edgecolor='red', facecolor='none')
    axes[1].add_patch(rect)
    cat_name = train_dataset.categories[label.item()]
    axes[1].text(x1, y1-10, f"{cat_name} {score:.2f}", color='red', fontsize=10, weight='bold')
axes[1].axis('off')

plt.tight_layout()
plt.show()

#### CARDD Change

In [ ]:
import os
os.listdir("../data/CarDD_release/CarDD_release/CarDD_COCO/annotations")

In [ ]:
import json

with open("../data/CarDD_release/CarDD_release/CarDD_COCO/annotations/instances_train2017.json") as f:
    train_annos = json.load(f)

print(train_annos.keys())
print("\nCategories:")
print(train_annos['categories'])
print("\nFirst image entry:")
print(train_annos['images'][0])
print("\nFirst annotation entry:")
print(train_annos['annotations'][0])
print(f"\nTotal images: {len(train_annos['images'])}")
print(f"Total annotations: {len(train_annos['annotations'])}")

In [ ]:
import pandas as pd

image_info = pd.read_excel("../data/CarDD_release/CarDD_release/CarDD_COCO/annotations/image_info.xlsx")
print(image_info.columns.tolist())
image_info.head()

In [ ]:
# Load CarDD annotations
with open("../data/CarDD_release/CarDD_release/CarDD_COCO/annotations/instances_train2017.json") as f:
    train_coco = json.load(f)

with open("../data/CarDD_release/CarDD_release/CarDD_COCO/annotations/instances_val2017.json") as f:
    val_coco = json.load(f)

with open("../data/CarDD_release/CarDD_release/CarDD_COCO/annotations/instances_test2017.json") as f:
    test_coco = json.load(f)

train_img_dir = "../data/CarDD_release/CarDD_release/CarDD_COCO/train2017"
val_img_dir = "../data/CarDD_release/CarDD_release/CarDD_COCO/val2017"
test_img_dir = "../data/CarDD_release/CarDD_release/CarDD_COCO/test2017"

train_dataset = DamageDataset(train_coco, train_img_dir, transform=get_transform(train=True))
val_dataset = DamageDataset(val_coco, val_img_dir, transform=get_transform(train=False))
test_dataset = DamageDataset(test_coco, test_img_dir, transform=get_transform(train=False))

print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False, collate_fn=collate_fn)

In [ ]:
num_classes = len(train_dataset.categories) + 1
print(f"Num classes (including background): {num_classes}")

images, targets = next(iter(train_loader))
print(f"Batch size: {len(images)}")
print(f"Image shape: {images[0].shape}")
print(f"First target boxes: {targets[0]['boxes']}")
print(f"First target labels: {targets[0]['labels']}")

#### Single Epoch Test
Testing new CarDD dataset, switched from kaggle dataset to this because of lack of data

In [ ]:
model = build_model(num_classes)
model = model.to(device)

params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)

import time
start = time.time()
train_loss = train_one_epoch(model, optimizer, train_loader, device)
elapsed = time.time() - start

print(f"Epoch 1 | Train Loss: {train_loss:.4f} | Time: {elapsed:.1f}s ({elapsed/60:.1f} min)")

In [ ]:
print(f"Device variable: {device}")
print(f"Model is on: {next(model.parameters()).device}")
print(f"CUDA available: {torch.cuda.is_available()}")

In [ ]:
model = build_model(num_classes)
model = model.to(device)

params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)

import time
start = time.time()
train_loss = train_one_epoch(model, optimizer, train_loader, device)
elapsed = time.time() - start

print(f"Epoch 1 | Train Loss: {train_loss:.4f} | Time: {elapsed:.1f}s ({elapsed/60:.1f} min)")

In [ ]:
import time

def train_one_epoch(model, optimizer, data_loader, device, print_every=10):
    model.train()
    total_loss = 0.0
    epoch_start = time.time()

    for batch_idx, (images, targets) in enumerate(data_loader):
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())

        optimizer.zero_grad()
        losses.backward()
        optimizer.step()

        total_loss += losses.item()

        if batch_idx % print_every == 0:
            elapsed = time.time() - epoch_start
            avg_time_per_batch = elapsed / (batch_idx + 1)
            batches_remaining = len(data_loader) - (batch_idx + 1)
            eta_seconds = avg_time_per_batch * batches_remaining
            print(f"  Batch {batch_idx}/{len(data_loader)} | Loss: {losses.item():.4f} | "
                  f"Elapsed: {elapsed:.1f}s | Avg: {avg_time_per_batch:.2f}s/batch | ETA: {eta_seconds/60:.1f} min")

    return total_loss / len(data_loader)

In [ ]:
model = build_model(num_classes)
model = model.to(device)

params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)

start = time.time()
train_loss = train_one_epoch(model, optimizer, train_loader, device)
elapsed = time.time() - start

print(f"\nEpoch 1 complete | Train Loss: {train_loss:.4f} | Total Time: {elapsed/60:.1f} min")

#### NOTES
Kaggle dataset will not be used anymore, and severity will most likely need to have to be accounted for using another dataset later down the pipeline. The CarDD dataset will be great for localized car parts and damage detection, and is a huge dataset. Much more involved than the previous dataset, so it will most likely take 2-3 hours for 8-10 epochs on my hardware.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

In [ ]:
print(f"Train dataset size: {len(train_dataset)}")
print(f"Number of batches: {len(train_loader)}")

#### Actual full training run

In [ ]:
def train_one_epoch(model, optimizer, data_loader, device, print_every=100):
    model.train()
    total_loss = 0.0
    epoch_start = time.time()

    for batch_idx, (images, targets) in enumerate(data_loader):
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())

        optimizer.zero_grad()
        losses.backward()
        optimizer.step()

        total_loss += losses.item()

        if batch_idx % print_every == 0:
            print(f"  Batch {batch_idx}/{len(data_loader)} | Loss: {losses.item():.4f}")

    epoch_time = time.time() - epoch_start
    print(f"  Epoch time: {epoch_time/60:.1f} min")

    return total_loss / len(data_loader)

In [ ]:
def evaluate_loss(model, data_loader, device):
    model.train()  # Mask R-CNN only returns losses in train mode
    total_loss = 0.0

    with torch.no_grad():
        for images, targets in data_loader:
            images = [img.to(device) for img in images]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

            loss_dict = model(images, targets)
            losses = sum(loss for loss in loss_dict.values())
            total_loss += losses.item()

    return total_loss / len(data_loader)

In [ ]:
#Sanity Check
print(build_model)
print(train_one_epoch)
print(evaluate_loss)
print(f"Train dataset size: {len(train_dataset)}")
print(f"Device: {device}")

In [ ]:
model = build_model(num_classes)
model = model.to(device)

params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)

os.makedirs("../models/damage_detector", exist_ok=True)
best_val_loss = float('inf')
num_epochs = 10

for epoch in range(num_epochs):
    print(f"\n=== Epoch {epoch+1}/{num_epochs} ===")
    train_loss = train_one_epoch(model, optimizer, train_loader, device)
    val_loss = evaluate_loss(model, val_loader, device)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "../models/damage_detector/maskrcnn_cardd_best.pth")
        print(f"  -> New best model saved (Val Loss: {val_loss:.4f})")

    print(f"Epoch {epoch+1}/{num_epochs} Summary | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

print(f"\nTraining complete. Best Val Loss: {best_val_loss:.4f}")

In [ ]:
model = build_model(num_classes)
model.load_state_dict(torch.load("../models/damage_detector/maskrcnn_cardd_best.pth"))
model = model.to(device)

test_loss = evaluate_loss(model, test_loader, device)
print(f"Test Loss: {test_loss:.4f}")

In [ ]:
model.roi_heads.nms_thresh = 0.3 

In [ ]:
model.eval()
confidence_threshold = 0.5
num_to_check = 5

fig, axes = plt.subplots(num_to_check, 2, figsize=(16, 8 * num_to_check))

for i in range(num_to_check):
    img, target = test_dataset[i]
    with torch.no_grad():
        prediction = model([img.to(device)])

    img_display = img.permute(1, 2, 0).cpu().numpy()
    keep = prediction[0]['scores'] > confidence_threshold
    pred_boxes = prediction[0]['boxes'][keep].cpu()
    pred_labels = prediction[0]['labels'][keep].cpu()
    pred_scores = prediction[0]['scores'][keep].cpu()

    axes[i, 0].imshow(img_display)
    axes[i, 0].set_title(f"Image {i} - Ground Truth")
    for box, label in zip(target['boxes'], target['labels']):
        x1, y1, x2, y2 = box
        rect = patches.Rectangle((x1, y1), x2-x1, y2-y1, linewidth=2, edgecolor='lime', facecolor='none')
        axes[i, 0].add_patch(rect)
        axes[i, 0].text(x1, y1-10, test_dataset.categories[label.item()], color='lime', fontsize=10, weight='bold')
    axes[i, 0].axis('off')

    axes[i, 1].imshow(img_display)
    axes[i, 1].set_title(f"Image {i} - Predictions (score > {confidence_threshold})")
    for box, label, score in zip(pred_boxes, pred_labels, pred_scores):
        x1, y1, x2, y2 = box
        rect = patches.Rectangle((x1, y1), x2-x1, y2-y1, linewidth=2, edgecolor='red', facecolor='none')
        axes[i, 1].add_patch(rect)
        cat_name = test_dataset.categories[label.item()]
        axes[i, 1].text(x1, y1-10, f"{cat_name} {score:.2f}", color='red', fontsize=10, weight='bold')
    axes[i, 1].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
model = build_model(num_classes)
model = model.to(device)

params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)

# Drop LR by 10x every 7 epochs
lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.1)

os.makedirs("../models/damage_detector", exist_ok=True)
best_val_loss = float('inf')
num_epochs = 15

for epoch in range(num_epochs):
    print(f"\n=== Epoch {epoch+1}/{num_epochs} ===")
    train_loss = train_one_epoch(model, optimizer, train_loader, device, print_every=100)
    val_loss = evaluate_loss(model, val_loader, device)
    lr_scheduler.step()

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "../models/damage_detector/maskrcnn_cardd_best.pth")
        print(f"  -> New best model saved (Val Loss: {val_loss:.4f})")

    current_lr = optimizer.param_groups[0]['lr']
    print(f"Epoch {epoch+1}/{num_epochs} Summary | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | LR: {current_lr}")

print(f"\nTraining complete. Best Val Loss: {best_val_loss:.4f}")

In [ ]:
model = build_model(num_classes)
model.load_state_dict(torch.load("../models/damage_detector/maskrcnn_cardd_best.pth"))
model = model.to(device)

test_loss = evaluate_loss(model, test_loader, device)
print(f"Test Loss: {test_loss:.4f}")

In [ ]:
model = build_model(num_classes)
model.load_state_dict(torch.load("../models/damage_detector/maskrcnn_cardd_best.pth"))
model = model.to(device)
model.roi_heads.nms_thresh = 0.3
model.eval()

confidence_threshold = 0.5
images_to_check = [0, 2, 3, 4]

fig, axes = plt.subplots(len(images_to_check), 2, figsize=(16, 8 * len(images_to_check)))

for row, i in enumerate(images_to_check):
    img, target = test_dataset[i]
    with torch.no_grad():
        prediction = model([img.to(device)])

    img_display = img.permute(1, 2, 0).cpu().numpy()
    keep = prediction[0]['scores'] > confidence_threshold
    pred_boxes = prediction[0]['boxes'][keep].cpu()
    pred_labels = prediction[0]['labels'][keep].cpu()
    pred_scores = prediction[0]['scores'][keep].cpu()

    axes[row, 0].imshow(img_display)
    axes[row, 0].set_title(f"Image {i} - Ground Truth")
    for box, label in zip(target['boxes'], target['labels']):
        x1, y1, x2, y2 = box
        rect = patches.Rectangle((x1, y1), x2-x1, y2-y1, linewidth=2, edgecolor='lime', facecolor='none')
        axes[row, 0].add_patch(rect)
        axes[row, 0].text(x1, y1-10, test_dataset.categories[label.item()], color='lime', fontsize=10, weight='bold')
    axes[row, 0].axis('off')

    axes[row, 1].imshow(img_display)
    axes[row, 1].set_title(f"Image {i} - Predictions (score > {confidence_threshold}, NMS 0.3)")
    for box, label, score in zip(pred_boxes, pred_labels, pred_scores):
        x1, y1, x2, y2 = box
        rect = patches.Rectangle((x1, y1), x2-x1, y2-y1, linewidth=2, edgecolor='red', facecolor='none')
        axes[row, 1].add_patch(rect)
        cat_name = test_dataset.categories[label.item()]
        axes[row, 1].text(x1, y1-10, f"{cat_name} {score:.2f}", color='red', fontsize=10, weight='bold')
    axes[row, 1].axis('off')

plt.tight_layout()
plt.show()